## Libraries

We use Python libraries to generate a synthetic SQLite database for a fitness tracker app. The libraries used are sqlite3 for database creation, random for numeric values, and mimesis for realistic names. No external datasets are required.

In [3]:
# Import libraries
import sqlite3
import random
from mimesis import Person
from mimesis.locales import Locale

# Initialize synthetic name generator
person = Person(Locale.EN)

## Database Creation

A SQLite database named fitness_tracker.db is created. Foreign key enforcement is turned on to maintain consistency between tables and ensure referential integrity.

In [4]:
# Connect to SQLite database
conn = sqlite3.connect("fitness_tracker.db")
cursor = conn.cursor()

# Enable foreign keys
cursor.execute("PRAGMA foreign_keys = ON")

## Users Table

The users table stores demographic and physical information for each user. A primary key user_id ensures uniqueness, while columns such as name, gender, age, experience_level, height_cm, and weight_kg capture the necessary attributes. For this dataset, 1100 users are generated.

In [6]:
# Create users table
cursor.execute("""
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    gender TEXT,
    age INTEGER,
    experience_level TEXT,  -- Beginner, Intermediate, Advanced
    height_cm REAL,
    weight_kg REAL
)
""")

# Generate users
users = []
for i in range(1100):
    users.append((
        i+1,
        person.full_name(),
        random.choice(['Male', 'Female', 'Other']),
        random.randint(15, 65),
        random.choice(['Beginner','Intermediate','Advanced']),
        round(random.uniform(150, 200),1),
        round(random.uniform(45, 120),1)
    ))

cursor.executemany("INSERT INTO users VALUES (?,?,?,?,?,?,?)", users)

## Workouts Table

The workouts table holds details of different workouts offered in the app. Each workout has a unique workout_id. Additional attributes include workout_name, category, intensity_level, and calories_burned_per_hour.

In [7]:
# Create workouts table
cursor.execute("""
CREATE TABLE IF NOT EXISTS workouts (
    workout_id INTEGER PRIMARY KEY,
    workout_name TEXT,
    category TEXT,          -- Strength, Cardio, Flexibility
    intensity_level TEXT,   -- Low, Medium, High
    calories_burned_per_hour REAL
)
""")

# Add workouts
workouts = [
    ('Running','Cardio','Medium',600),
    ('Cycling','Cardio','Medium',500),
    ('Yoga','Flexibility','Low',200),
    ('Weight Lifting','Strength','High',450),
    ('Swimming','Cardio','High',700),
    ('Pilates','Flexibility','Medium',250),
    ('HIIT','Cardio','High',750)
]

cursor.executemany(
    "INSERT INTO workouts (workout_name, category, intensity_level, calories_burned_per_hour) VALUES (?,?,?,?)",
    workouts
)

## Logs Table

The logs table records workout sessions for each user. The composite primary key (user_id, workout_id, date) ensures no duplicate entries for the same workout on the same day. Foreign keys maintain referential integrity with users and workouts.

In [ ]:
# Create logs table
cursor.execute("""
CREATE TABLE IF NOT EXISTS logs (
    user_id INTEGER,
    workout_id INTEGER,
    date TEXT,
    duration_min INTEGER,
    calories_burned REAL,
    PRIMARY KEY(user_id, workout_id, date),
    FOREIGN KEY(user_id) REFERENCES users(user_id),
    FOREIGN KEY(workout_id) REFERENCES workouts(workout_id)
)
""")

## Data Generation – Users

Synthetic user data is generated using mimesis for names and random for numeric attributes. Each user is assigned a gender, age, experience level, height, and weight.

In [8]:
# Create logs table
cursor.execute("""
CREATE TABLE IF NOT EXISTS logs (
    user_id INTEGER,
    workout_id INTEGER,
    date TEXT,
    duration_min INTEGER,
    calories_burned REAL,
    PRIMARY KEY(user_id, workout_id, date),
    FOREIGN KEY(user_id) REFERENCES users(user_id),
    FOREIGN KEY(workout_id) REFERENCES workouts(workout_id)
)
""")

## Data Generation – Workouts

Seven predefined workouts are added to the workouts table. Categories include Strength, Cardio, and Flexibility, with varying intensity levels and calories burned per hour.

In [7]:
# Add Workouts
workouts = [
    ('Running','Cardio','Medium',600),
    ('Cycling','Cardio','Medium',500),
    ('Yoga','Flexibility','Low',200),
    ('Weight Lifting','Strength','High',450),
    ('Swimming','Cardio','High',700),
    ('Pilates','Flexibility','Medium',250),
    ('HIIT','Cardio','High',750)
]

cursor.executemany(
    "INSERT INTO workouts (workout_name, category, intensity_level, calories_burned_per_hour) VALUES (?,?,?,?)",
    workouts
)

## Data Generation – Logs

Each user is randomly assigned 5–12 workout sessions. Duration is between 20–120 minutes, and calories are calculated based on the workout type and duration with slight randomness.

In [ ]:
# Create logs table
cursor.execute("""
CREATE TABLE IF NOT EXISTS logs (
    user_id INTEGER,
    workout_id INTEGER,
    date TEXT,
    duration_min INTEGER,
    calories_burned REAL,
    PRIMARY KEY(user_id, workout_id, date),
    FOREIGN KEY(user_id) REFERENCES users(user_id),
    FOREIGN KEY(workout_id) REFERENCES workouts(workout_id)
)
""")

# Generate logs
logs = []

for user_id in range(1, 1101):
    num_sessions = random.randint(5,12)
    used_combinations = set()  # track (workout_id, date)

    for _ in range(num_sessions):
        while True:
            workout_id = random.randint(1, len(workouts))
            date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
            combo = (workout_id, date)
            if combo not in used_combinations:
                used_combinations.add(combo)
                break  # unique combo

        duration = random.randint(20, 120)
        calories = round(duration/60 * workouts[workout_id-1][3] * random.uniform(0.8,1.2),1)
        logs.append((user_id, workout_id, date, duration, calories))

cursor.executemany("INSERT INTO logs VALUES (?,?,?,?,?)", logs)